> 前缀 $(x, y_{\lt t})$
- Pretrain-SFT-RL-OPD
    - SFT 在预训练之后通常是必要的,因为没有基本指令遵循和格式能力,RL 很难高效启动。问题在于 SFT 之后如何继续:数学和代 码任务奖励更可验证,往往偏向 RLVR;创意写作和知识密集任务的 reward 更嘈杂,distillation 或  self-distillation 往往更自然。
    - 可验证任务:优先考虑 RL/RLVR,因为 reward 偏差较低。
    - 高主观任务:distillation 更可控,但要警惕 teacher bias 和风格坍缩。
    - 专家合并:OPD/MOPD 的吸引力在于,它允许用多个 expert teacher 的局部建议塑造 student,  而不是让 final checkpoint 自己再次经历完整 RL。
- SFT 常 常在外部数据前缀上训练;
    - 当任务的输出格式、指令遵循习惯或基本行为还不存在时,SFT 的“外部分布拉动”很有效。它 可以快速把模型带到一个新的行为区域,例如从纯续写模型变成能遵守 instruction 的模型。
- RL 和 OPD 都让模型先走自己的路径,再在这些路径上改概率。因此二者更容易修正模型自己会犯的错,而不是只模仿另一个系统的标准答案。
- 遗忘
    - 一个模型原来会写代码、会遵循格式、会做常见推理,这些能力 对应很多已经学好的 token 概率习惯。新训练如果要求它大量模仿另一种写法,就像把很多原来熟悉 的岔路口重新调了路牌:新任务可能学会了,但旧任务里本来正确的续写概率也被改低了。
    - 训练如果 总是在模型自己生成的前缀上发生,改动更像是在它已经会走的路上修路,而不是把它整条路线搬到 另一个地图上。

### SFT

- 训练样本 $(x, y)$，其中 $y=(y_1,\cdots,y_T)$，自回归语言模型把整段 completion 的概率分解为逐 token 概率的乘积:
    $$
    \pi_\theta(y|x)=\Pi_{t=1}^T\pi_\theta(y_t|x,y_{\lt t})
    $$
  - $y_t$：第 $t$ 个目标
  - $y_{\lt t}$：目标答案中第 $t$ 个 token 之前的前缀（gt，teacher forcing，非 model sample generated）。
  - $\pi_\theta(y_t|x,y_{\lt t})$：模型在这个前缀后面给正确 token 的概率。
- 最大似然训练要让整段答案的概率尽量大,也就是最大化 $\log\pi_\theta(y|x)$，SFT loss：neg log-likelihood

$$
\begin{split}
\log\pi_\theta(y|x)&=\sum_{t=1}^T\log\pi_\theta(y_t|x,y_{\lt t})\\
\mathcal L_{sft}(\theta)&=\mathbb E_{x,y\sim D}\left[-\sum_{t=1}^T\log\pi_\theta(y_t|x,y_{\lt t})\right]
\end{split}
$$

In [1]:
import numpy as np

In [3]:
-np.log(0.5), -np.log(0.05)

(np.float64(0.6931471805599453), np.float64(2.995732273553991))

- $-\log p$：模型越不相信数据集里的下一个 token,罚分越大。

如果把数据集中同一个前缀后面可能出现的下一个 token 统计成一个经验分布 $q_{D}(\cdot|x,y_{\lt t})$，上式 就是交叉熵:

$$
H(q_D,\pi_\theta)=-\sum_{v\in V} q_D(v|x,y_{\lt t})\log\pi_\theta(v|x,y_{\lt t})
$$

- $q_D(v|x,y_{\lt t})$: 数据集认为这个位置应该写 $v$ 的频率;普通单答案 SFT 里,它几乎就是 one-hot,  正确 token 为 1,其他 token 为 0。

$$
\begin{split}
\text{KL}(q_D||\pi_\theta)&= \sum_{v \in V} q_D(v) \log \frac{q_D(v)}{\pi_\theta(v)} = \sum_{v \in V} q_D(v) \log q_D(v) - \sum_{v \in V} q_D(v) \log \pi_\theta(v)\\
&=-H(q_D)- \sum_{v \in V} q_D(v) \log \pi_\theta(v)\\
\end{split}
$$

- $H(q_D)$ 只由数据分布决定,不随模型参数改变:
$$
H(q_D,\pi_\theta)=\text{KL}(q_D||\pi_\theta)+H(q_D)
$$

### RL

KL-constrained update 可以把这个“先不离原模型太远,再提高奖励”的想法写成一个小优化问题。固定一个 prompt, 先把旧策略 $\pi_{old}$，成上一轮模型的下一个 token/序列分布;我们要找一个新分布 $\pi$，它一方面期望奖励高,另一方面不要和旧分布差太远:

$$
\max_{\pi} \left[ \sum_y \pi(y)R(y) - \frac{1}{\beta}\text{KL}(\pi||\pi_{\text{old}}) \right], \quad \text{s.t.} \quad \sum_y \pi(y) = 1.
$$

把 KL 展开,并加上归一化约束的拉格朗日乘子 $\lambda$:

$$
\mathcal{J}(\pi, \lambda) = \sum_y \pi(y)R(y) - \frac{1}{\beta} \sum_y \pi(y) \log \frac{\pi(y)}{\pi_{\text{old}}(y)} + \lambda \left( \sum_y \pi(y) - 1 \right).
$$

对每个 $y$ 求偏导并令其为 0:

$$
\frac{\partial \mathcal{J}}{\partial \pi(y)} = R(y) - \frac{1}{\beta} \left[ \log \frac{\pi(y)}{\pi_{\text{old}}(y)} + 1 \right] + \lambda = 0.
$$

$$
\pi^*(y) = \frac{1}{Z(x)} \pi_{\text{old}}(y|x) \exp(\beta R(y))
$$

RL 不是凭空跳到一个外部答案集,而是在旧模型本来会写的 答案上乘一个奖励倍率。旧模型原本概率几乎为 0 的答案,即使奖励很高,也很难在一次小步更新里 突然获得大量概率;旧模型本来常写、且奖励较高的答案,会更快变常见。也因此,当奖励是可靠的, 比如 RLVR 中的可验证数学/代码奖励,RL 往往能从模型已有能力附近改出更好的答案。
- $Z(x)$: 归一化常数,使所有 $y$ 的概率加起来为1
- $\exp(\beta R(y))$: 像一个倍率。高奖励答案乘上大倍率,低奖励答案乘上小倍率。

mirror update（Mirror Descent）

------

$\frac{\partial^2 \mathcal{J}}{\partial \pi(y)^2} = \frac{\partial}{\partial \pi(y)} \left( R(y) - \frac{1}{\beta} \left[ \log \frac{\pi(y)}{\pi_{\text{old}}(y)} + 1 \right] + \lambda \right) = - \frac{1}{\beta \pi(y)}\lt 0$

Hessian 矩阵负定意味着原函数是一个严格凹函数 (Strictly Concave Function)。对于严格凹函数而言，只要找到一阶导数为 0 的点，它就必然是唯一的全局最大值点。

RLVR 的奖励来自可验证结果,例如答案正确、代码测试通过,偏差通常更低;RLHF 的 reward model 是学习出来的偏好代理,可能带有系统性偏差。因此 RLHF 更依赖 KL penalty、PPO clipping、trust region 等机制,防止模型过度优化一个不完美代理

### 高主观任务里,teacher 从哪里来

- 在数学和代码任务里, teacher 可以来自 RLVR 训练出的 expert,也可以来自能稳定给出正确答案的大模型。
- 强模型 teacher:用更大的通用模型或专门模型,在相同 student 前缀上给 logits 或 logprobs。优点是覆盖面广;风险是它会把自己的风格、长度偏好和措辞习惯一起传给 student。
- 专家 teacher:先用人工精选数据、偏好数据或任务 rubric 训练一个专门 teacher,再用 OPD 把能力迁到目标 student。适合客服、法律摘要、医学信息抽取、企业内部助手等有明确规范的任务。
- 多 teacher 合并:不同领域或不同风格各有一个 teacher,最终 student 在自己的 rollouts 上接受多个 teacher 的局部建议。MiMo/MOPD 一类专家合并就属于这个方向。